In [1]:
   import pandas as pd

   df = pd.read_csv("churn_data.csv")
   df.head()

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [2]:
df.shape # (nombre de lignes, nombre de colonnes) -> doit afficher (500, 6)

(500, 6)

In [3]:
df.describe() # count (500 clients), moyenne, min, max, écart-type, 25%, 50%, 75% des clients

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [4]:
df.isnull().sum() # compte les valeurs manquantes par colonne -> les 500 lignes sont complètes, aucune donnée n'est vide, doit être 0 partout

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [5]:
df["churn"].value_counts() # compte les clients "0" (restent) vs "1" (partent)

churn
0    481
1     19
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

X = df[
    [
        "age",
        "salaire",
        "abonnement",
        "anciennete",
        "consommation"
    ]
]

y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
X_train.shape

(400, 5)

In [8]:
X_test.shape

(100, 5)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [10]:
import joblib

joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [11]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl")
print("model.pkl écrasé avec l'Essai 2")

# ---------- ESSAI 3 : on retire la colonne "salaire" ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl")
print("model.pkl écrasé avec l'Essai 3")

# ---------- Vérification ----------
loaded = joblib.load("model.pkl")
print("Nombre de features attendu :", loaded.n_features_in_)

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2
Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3
Nombre de features attendu : 4


## Questions de réflexion

Essai 1 : 0.99 / Essai 2 : 0.99 / Essai 3 : 1.0

**1. Quel est le meilleur des 3 modèles ? Peux-tu le prouver avec certitude ?**
A priori c'est l'essai 3 (1.0), mais je ne peux pas le prouver. Les résultats sont juste affichés dans le notebook, rien n'est enregistré, et la différence c'est seulement 1 client sur 100. En plus le 1.0 vient surtout de la fuite de données.

**2. Peux-tu redonner exactement le modèle de l'Essai 1 ?**
Non, model.pkl a été écrasé par l'essai 2 puis par l'essai 3. Il contient seulement l'essai 3 (4 features).

**3. Un camarade avec mon CSV et mon code obtiendra-t-il exactement le même résultat ?**
Pas forcément. Les seeds sont fixés mais les versions de Python et des librairies ne sont pas fixées, et il peut utiliser un autre kernel.